In [17]:
import torch
from torch import nn
from torch.nn import functional as F

In [18]:
# Constant weight와 control flow를 가진 Module

class FixedHiddenMLP(nn.Module):

    def __init__(self):
        super().__init__()

        # Gradient를 계산하지 않는 constant tensor
        self.rand_weight = torch.rand(
            (20, 20)
        )

        self.linear = nn.LazyLinear(20)


def fixed_hidden_mlp_forward(self, X):

    # [B, 20] -> [B, 20]
    X = self.linear(X)

    # [B, 20] @ [20, 20] -> [B, 20]
    X = F.relu(
        X @ self.rand_weight + 1
    )

    # 같은 Linear layer와 parameter를 재사용
    X = self.linear(X)

    # Python control flow
    while X.abs().sum() > 1:
        X /= 2

    return X.sum()


FixedHiddenMLP.forward = (
    fixed_hidden_mlp_forward
)

In [19]:
# FixedHiddenMLP 실행 확인

# [2, 20]
X = torch.rand(2, 20)

net = FixedHiddenMLP()
Y = net(X)

print(net)
print(Y)
print(Y.shape)

FixedHiddenMLP(
  (linear): Linear(in_features=20, out_features=20, bias=True)
)
tensor(-0.1335, grad_fn=<SumBackward0>)
torch.Size([])


In [20]:
# 여러 Module을 중첩한 model

class NestMLP(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.LazyLinear(32),
            nn.ReLU(),
        )
        
        self.linear = nn.LazyLinear(16)
        
        
def nest_mlp_forward(self, X):

    # [B, 20] -> [B, 64] -> [B, 32] -> [B, 16]
    return self.linear(
        self.net(X)
    )


NestMLP.forward = nest_mlp_forward

chimera = nn.Sequential(
    # [B, 20] -> [B, 64] -> [B, 32] -> [B, 16]
    NestMLP(), 
    
    # [B, 16] -> [B, 20]
    nn.LazyLinear(20),
    
    # [B, 20] -> scalar
    FixedHiddenMLP(),
)

print(chimera)
print(Y)
print(Y.shape)

Sequential(
  (0): NestMLP(
    (net): Sequential(
      (0): LazyLinear(in_features=0, out_features=64, bias=True)
      (1): ReLU()
      (2): LazyLinear(in_features=0, out_features=32, bias=True)
      (3): ReLU()
    )
    (linear): LazyLinear(in_features=0, out_features=16, bias=True)
  )
  (1): LazyLinear(in_features=0, out_features=20, bias=True)
  (2): FixedHiddenMLP(
    (linear): LazyLinear(in_features=0, out_features=20, bias=True)
  )
)
tensor(-0.1335, grad_fn=<SumBackward0>)
torch.Size([])
